In [1]:
import pyrosetta
import numpy as np

from benchmark import bpti_ryfyn_benchmark
from energy_calculation import evaluate_quantum_energies, evaluate_pyrosetta_energies, compare_energies
from misc import init_generator_params
from rotamer_extraction import extract_top_n_rotamers
from h_ising_creation import extract_hamiltonian_tensors, build_ising_hamiltonian, reduce_hamiltonian
from initialisation import initialize_rosetta
from custom_qaoa import qaoa_func_generator, run_qaoa
from h_mixer import custom_xy_mixer_layer

from constants import *
from validation import validate_conformations

initialize_rosetta(pyrosetta, extra_flags="-mute all")

Initializing PyRosetta with cleaning flags: -ignore_unrecognized_res and extra flags: -mute all
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.m1 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org


In [3]:
# Pyrosetta Relevant Code
benchmark_pose = bpti_ryfyn_benchmark()
rotamer_lib, ig, rot_sets = extract_top_n_rotamers(benchmark_pose)

# Generating QUBO (Quadratic Unconstrained Binary Optimisation) Model, and then reduce it
h_linear, J_quadratic = extract_hamiltonian_tensors(rotamer_lib, ig, rot_sets)
h_flex_linear, J_flex_quadratic, global_offset = reduce_hamiltonian(h_linear, J_quadratic, rotamer_lib)
generator_params = init_generator_params(h_flex_linear)
for idx in h_linear:
    print(h_linear[idx])
    print(h_flex_linear.get(idx, "None"))
    print("\n---------------------------------------\n")

# Generate the actual observable and running functions we will use in the QAOA Algorithm
H_ising = build_ising_hamiltonian(h_flex_linear, J_flex_quadratic, global_offset, penalty=0.0)
cost_function, sample_function = qaoa_func_generator(H_ising, custom_xy_mixer_layer, generator_params)

# Run the Quantum Approximate Optimisation Algorithm and sample the final parameters
final_params = run_qaoa(cost_function)
probabilities = sample_function(final_params)

# Extract the top 100 most probably conformations and check that exactly 1 rotamer for each residue is selected
top_indices = list(np.argsort(probabilities)[-TOP_CONFORMATION_COUNTS:][::-1])
valid_conformations = validate_conformations(top_indices, probabilities, generator_params)

Initializing PyRosetta with cleaning flags: -ignore_unrecognized_res and extra flags: -mute all
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.m1 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org
Fragment Sequence: RYFYN
Total Residues: 5
Creating score func

KeyError: 'energy'

In [ ]:
def evaluate_quantum_energies(valid_conformations, h_flex, J_flex, global_offset, params):
    wire_offsets = params["wire_offsets"]

    for conformation in valid_conformations:
        bitstring = conformation["bitstring"]
        current_energy = global_offset

        # One body energies
        for seq, energies in h_flex.items():
            base_wire = wire_offsets[seq]
            for rot, e_val in enumerate(energies):
                if bitstring[base_wire + rot] == 1:
                    current_energy += e_val

        # Two body energies
        for (seq_i, seq_j), interactions in J_flex.items():
            for (rot_i, rot_j), e_val in interactions.items():
                k = wire_offsets[seq_i] + rot_i
                l = wire_offsets[seq_j] + rot_j

                if bitstring[k] == 1 and bitstring[l] == 1:
                    current_energy += e_val

        conformation['quantum_energy'] = current_energy
    # raise NotImplementedError("Not yet implemented")

evaluate_quantum_energies(valid_conformations, h_flex_linear, J_flex_quadratic, global_offset, params=generator_params)

In [ ]:
def evaluate_pyrosetta_energies(pose, bitstrings, rotsets):
    raise NotImplementedError("Not yet implemented")
